# <font color="#418FDE" size="6.5" uppercase>**Dichte Keras-Modelle**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Erstellen Sequential-Modelle mit Dense-Schichten und passenden Eingabe- und Ausgabeformen. 
- Trainieren kleine Keras-Modelle mit compile, fit, Validierungsdaten und History. 
- Bewerten, regularisieren, speichern und vergleichen Keras-Modelle mit Baselines. 


## **1. Keras Modellaufbau**

### **1.1. Sequential Modelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_01_01.jpg?v=1784810344" width="250">



>* Sequential-Modelle leiten Daten schichtweise weiter
>* Dense-Schichten eignen sich für tabellarische Merkmale

>* Klare lineare Reihenfolge der Schichten
>* Ungeeignet für Verzweigungen oder mehrere Eingänge

>* Eingabeform, Schichtgrößen und Ausgabe abstimmen
>* Lernkapazität passend zur Aufgabe wählen



In [ ]:
#@title Python-Code - Sequential Modelle

# Dieses Beispiel baut ein kleines Sequential-Modell nach.
# Dense-Schichten werden als einfache Matrixformen dargestellt.
# Die Ausgabe zeigt passende Eingabe- und Ausgabeformen.

import numpy as np

# Wir erzeugen drei Beispiele mit vier Eingabemerkmalen.
features = np.array(
    [[0.2, 1.0, 0.5, 0.0], [0.9, 0.1, 0.3, 0.7], [0.4, 0.8, 0.2, 0.6]]
)

# Jede Dense-Schicht hat Gewichte und Bias-Werte.
first_weights = np.array(
    [[0.4, -0.2, 0.1], [0.3, 0.5, -0.4], [-0.6, 0.2, 0.7], [0.1, -0.3, 0.2]]
)

first_bias = np.array([0.1, 0.0, -0.1])

# Die zweite Dense-Schicht erzeugt eine einzelne Ausgabe.
second_weights = np.array([[0.8], [-0.5], [0.3]])
second_bias = np.array([0.2])

# ReLU setzt negative Zwischenwerte auf null.
hidden_linear = features @ first_weights + first_bias
hidden_output = np.maximum(hidden_linear, 0)

# Die letzte Schicht wandelt drei Neuronen in eine Ausgabe um.
model_output = hidden_output @ second_weights + second_bias

# Diese Prüfungen machen die erwarteten Formen ausdrücklich sichtbar.
if features.shape[1] != first_weights.shape[0]:
    raise ValueError("Die Eingabeform passt nicht zur ersten Dense-Schicht.")

if hidden_output.shape[1] != second_weights.shape[0]:
    raise ValueError("Die verborgene Schicht passt nicht zur Ausgabeschicht.")

print("Sequential-Datenfluss mit Dense-Schichten")
print(f"Eingabe: {features.shape} = 3 Beispiele, 4 Merkmale")
print(f"Dense 1: {first_weights.shape} -> Ausgabe {hidden_output.shape}")
print(f"Dense 2: {second_weights.shape} -> Ausgabe {model_output.shape}")
print(f"Erste Vorhersage: {round(float(model_output[0, 0]), 3)}")



### **1.2. Eingabeformen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_01_02.jpg?v=1784810346" width="250">



>* Eingabeform beschreibt Merkmale eines einzelnen Beispiels
>* Stabile Struktur verhindert typische Dimensionsfehler

>* Zeilen sind Beispiele, Spalten sind Merkmale
>* Vorverarbeitung macht Eingaben numerisch und konsistent

>* Einzelbeispiel-Form bleibt, Batchgröße variiert
>* Mehrdimensionale Daten bewusst vor Dense abflachen



In [ ]:
#@title Python-Code - Eingabeformen verstehen

# Dieses Beispiel zeigt Eingabeformen für Dense-Modelle.
# Jede Zeile ist ein einzelnes Datenbeispiel.
# Die Ausgabe erklärt Batchgröße und Merkmalsanzahl.

import numpy as np

# Wir bauen kleine tabellarische Beispieldaten.
feature_names = ["Wohnfläche", "Zimmer", "Baujahr", "Entfernung", "Effizienz"]

# Jede Zeile beschreibt ein Haus mit fünf Merkmalen.
house_features = np.array(
    [[80, 3, 1998, 4.2, 0.82], [120, 5, 2010, 8.5, 0.91], [65, 2, 1985, 2.1, 0.70]],
    dtype=float,
)

# Die Zielwerte sind hier nur zur Einordnung vorhanden.
prices = np.array([320000, 510000, 280000], dtype=float)

# Eine einfache Prüfung macht die Form bewusst.
if house_features.ndim != 2:
    raise ValueError("Die Eingaben müssen eine Tabelle mit Zeilen und Spalten sein.")

# Keras würde nur die Merkmalsanzahl pro Beispiel benötigen.
batch_size = house_features.shape[0]
feature_count = house_features.shape[1]
input_shape_for_dense = (feature_count,)

# Ein einzelnes Beispiel ist ein Vektor gleicher Länge.
first_example = house_features[0]

print(f"Datenform des ganzen Stapels: {house_features.shape}")
print(f"Anzahl Beispiele im Stapel: {batch_size}")
print(f"Anzahl Merkmale pro Beispiel: {feature_count}")
print(f"Eingabeform für eine Dense-Schicht: {input_shape_for_dense}")
print(f"Erstes Beispiel hat Form: {first_example.shape}")
print(f"Merkmalsnamen: {', '.join(feature_names)}")
print(f"Zielwerte haben Form: {prices.shape}")



### **1.3. Passende Ausgabeschichten**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_01_03.jpg?v=1784810348" width="250">



>* Ausgabeform passend zur Aufgabe wählen
>* Regression: Neuronen entsprechen Zielwerten

>* Binär: ein Neuron für positive Klasse
>* Mehrklassen und Mehrfachlabel brauchen passende Neuronen

>* Ausgabe muss zu Zielcodierung passen
>* Vorhersageart bestimmt Neuronen und Aktivierung



In [ ]:
#@title Python-Code - Passende Ausgabeschichten

# Dieses Beispiel vergleicht passende Ausgabeschichten.
# Die Ausgabeform folgt direkt aus der Aufgabe.
# Am Ende siehst du klare Modellentscheidungen.

import numpy as np

# Jede Aufgabe beschreibt Zielwerte und passende letzte Schichten.
tasks = [
    ("Regression", "Wohnungspreis", (1,), "Dense(1)", "linear"),
    ("Binärklassifikation", "Spam ja/nein", (1,), "Dense(1)", "sigmoid"),
    ("Mehrklassenklassifikation", "Ziffer 0 bis 9", (10,), "Dense(10)", "softmax"),
    ("Mehrfachlabel", "Themen pro Artikel", (4,), "Dense(4)", "sigmoid"),
]

# Kleine Beispielvorhersagen zeigen die Bedeutung der Aktivierungen.
raw_outputs = np.array([-1.2, 0.3, 2.1, 0.8])
sigmoid_outputs = 1 / (1 + np.exp(-raw_outputs))
softmax_outputs = np.exp(raw_outputs) / np.exp(raw_outputs).sum()

# Diese Prüfung macht die wichtigste Formregel sichtbar.
for task_name, example, target_shape, layer, activation in tasks:
    output_units = target_shape[0]
    is_valid = output_units == target_shape[0]
    status = "passt" if is_valid else "passt nicht"
    print(f"{task_name}: Ziel {target_shape} -> {layer}, {activation}: {status}")

# Die Zahlen bleiben kurz und gut interpretierbar.
print(f"Sigmoid-Beispiel: {np.round(sigmoid_outputs[:3], 2).tolist()}")
print(f"Softmax-Summe: {round(float(softmax_outputs.sum()), 2)}")
print("Merksatz: Neuronen, Aktivierung und Zielcodierung müssen zusammenpassen.")



## **2. Training mit Keras**

### **2.1. Modell kompilieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_02_01.jpg?v=1784810350" width="250">



>* Kompilieren gibt dem Modell seine Lernaufgabe
>* Loss, Optimierer und Metriken steuern Training

>* Verlustfunktion passend zur Aufgabe wählen
>* Optimierer steuert Lerntempo und Stabilität

>* Metriken passend wählen, Genauigkeit kritisch prüfen
>* Kompilieren verbindet Architektur mit Training



### **2.2. Modelltraining mit fit**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_02_02.jpg?v=1784810352" width="250">



>* fit startet das Lernen mit Trainingsdaten
>* Vorhersagen verbessern sich durch Gewichtsänderungen

>* Epochen durchlaufen alle Trainingsdaten einmal
>* Batches und Epochen steuern stabiles Lernen

>* Validierungsdaten zeigen Lernen auf neuen Beispielen
>* History speichert Verlust und Metriken



### **2.3. Trainingsverlauf verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_02_03.jpg?v=1784810355" width="250">



>* History zeigt Kennzahlen über alle Epochen
>* Validierung erkennt Auswendiglernen statt Verallgemeinerung

>* Sinkende Verluste zeigen nützliches Lernen.
>* Verläufe erkennen Überanpassung und Trainingsprobleme.

>* Metriken immer im Anwendungskontext deuten
>* Trainingsverlauf zeigt sinnvolle Trainingsdauer



In [ ]:
#@title Python-Code - Trainingsverlauf verstehen

# Dieses Beispiel macht Trainingsverläufe sichtbar.
# Wir simulieren History-Werte aus Keras.
# Die Kurven zeigen Lernen und Überanpassung.

import numpy as np
import matplotlib.pyplot as plt

# Feste Zufallszahlen machen das Beispiel reproduzierbar.
rng = np.random.default_rng(42)
epochs = np.arange(1, 21)

# Trainingsverlust sinkt fast durchgehend über die Epochen.
train_loss = 0.9 * np.exp(-epochs / 7) + 0.08
train_loss = train_loss + rng.normal(0, 0.01, size=epochs.size)

# Validierungsverlust sinkt zuerst und steigt später wieder.
val_loss = 0.75 * np.exp(-epochs / 6) + 0.16
val_loss = val_loss + 0.012 * np.maximum(epochs - 10, 0)
val_loss = val_loss + rng.normal(0, 0.01, size=epochs.size)

# Eine History ähnelt dem Rückgabewert von model.fit.
history = {"loss": train_loss, "val_loss": val_loss}
best_epoch = int(epochs[np.argmin(history["val_loss"])])

# Kurze Ausgaben verbinden Zahlen mit Interpretation.
print(f"Epochen in der History: {len(epochs)}")
print(f"Bester Validierungsverlust in Epoche: {best_epoch}")
print("Danach kann weiteres Training auf Überanpassung hindeuten.")

# Eine einzelne Grafik zeigt Training und Validierung gemeinsam.
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs, history["loss"], marker="o", label="Trainingsverlust")
ax.plot(epochs, history["val_loss"], marker="o", label="Validierungsverlust")

# Die beste Epoche wird als Orientierung markiert.
ax.axvline(best_epoch, color="gray", linestyle="--", label="Beste Epoche")
ax.set_title("Trainingsverlauf verstehen")
ax.set_xlabel("Epoche")
ax.set_ylabel("Verlust")

# Legende und Raster erleichtern den Vergleich.
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



## **3. Bewertung und Vergleich**

### **3.1. Regularisierung gegen Overfitting**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_03_01.jpg?v=1784810357" width="250">



>* Overfitting verschlechtert Vorhersagen auf neuen Daten
>* Regularisierung begrenzt zu komplexe Modellregeln

>* Gewichte und Dropout verhindern dominante Neuronen
>* Kleinere Modelle generalisieren oft stabiler

>* Regularisierung mit Validierung und Baselines prüfen
>* Stabile Generalisierung wichtiger als Trainingsglanz



In [ ]:
#@title Python-Code - Regularisierung gegen Overfitting

# Dieses Beispiel zeigt Regularisierung gegen Overfitting.
# Wir vergleichen einfache Modelle mit gleicher Datenbasis.
# Die Validierungsleistung zeigt den Nutzen der Zurückhaltung.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Ein kleiner Datensatz enthält absichtlich etwas Rauschen.
features, target = make_classification(
    n_samples=900,
    n_features=20,
    n_informative=6,
    n_redundant=4,
    flip_y=0.18,
    class_sep=0.8,
    random_state=42,
)

# Die Aufteilung trennt Training und Validierung sauber.
X_train, X_valid, y_train, y_valid = train_test_split(
    features,
    target,
    test_size=0.35,
    stratify=target,
    random_state=42,
)

# Eine kurze Prüfung macht die Annahmen sichtbar.
if X_train.shape[0] == 0 or X_valid.shape[0] == 0:
    raise ValueError("Training und Validierung brauchen Beispiele.")

# Die Baseline sagt immer die häufigste Trainingsklasse voraus.
majority_class = np.bincount(y_train).argmax()
baseline_valid = accuracy_score(y_valid, np.full_like(y_valid, majority_class))

# Schwache Regularisierung erlaubt größere Gewichte.
weak_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=100.0, max_iter=1000, random_state=42),
)

# Stärkere Regularisierung begrenzt die Gewichte deutlicher.
regularized_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.08, max_iter=1000, random_state=42),
)

# Beide Modelle lernen nur aus den Trainingsdaten.
weak_model.fit(X_train, y_train)
regularized_model.fit(X_train, y_train)

# Wir messen Training und Validierung getrennt.
weak_train = accuracy_score(y_train, weak_model.predict(X_train))
weak_valid = accuracy_score(y_valid, weak_model.predict(X_valid))
regularized_train = accuracy_score(y_train, regularized_model.predict(X_train))
regularized_valid = accuracy_score(y_valid, regularized_model.predict(X_valid))

# Die Gewichtsnorm zeigt die Stärke der Zurückhaltung.
weak_norm = np.linalg.norm(weak_model.named_steps["logisticregression"].coef_)
regularized_norm = np.linalg.norm(
    regularized_model.named_steps["logisticregression"].coef_
)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Baseline Validierung: {baseline_valid:.3f}")
print(f"Schwach reguliert: Training {weak_train:.3f}, Validierung {weak_valid:.3f}")
print(f"Stärker reguliert: Training {regularized_train:.3f}, Validierung {regularized_valid:.3f}")
print(f"Gewichtsnorm schwach/stärker: {weak_norm:.2f} / {regularized_norm:.2f}")

# Das Diagramm vergleicht die entscheidenden Kennzahlen.
labels = ["Baseline", "Schwach", "Stärker"]
valid_scores = [baseline_valid, weak_valid, regularized_valid]

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(labels, valid_scores, color=["gray", "tomato", "seagreen"])
ax.set_title("Validierungsgenauigkeit mit und ohne Regularisierung")
ax.set_xlabel("Vergleichsmodell")
ax.set_ylabel("Genauigkeit")
ax.set_ylim(0, 1)
plt.show()



### **3.2. Vorhersagen interpretieren**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_03_02.jpg?v=1784810359" width="250">



>* Vorhersagen im fachlichen Kontext deuten
>* Konfidenz und Unsicherheit gemeinsam bewerten

>* Regressionswerte mit Fehlern und Vergleichswerten prüfen
>* Ungewöhnliche Fälle fachlich kritisch hinterfragen

>* Einzelfälle zeigen Modellnutzen jenseits von Metriken
>* Baselines vergleichen, Vertrauen und Prüfbedarf erkennen



In [ ]:
#@title Python-Code - Vorhersagen interpretieren

# Dieses Beispiel interpretiert einzelne Klassifikationsvorhersagen.
# Konfidenz zeigt Sicherheit und mögliche Grenzfälle.
# Baseline und Modell werden direkt verglichen.

import numpy as np
import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine kurze Prüfung verhindert unklare Formen.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Der Split bleibt reproduzierbar und erhält Klassenanteile.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Die Baseline lernt keine Muster, nur die häufigste Klasse.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Das Modell skaliert Merkmale und lernt eine einfache Grenze.
model = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Kennzahlen zeigen den Überblick, aber nicht einzelne Unsicherheiten.
baseline_accuracy = accuracy_score(y_test, baseline.predict(X_test))
model_accuracy = accuracy_score(y_test, model.predict(X_test))

# Wahrscheinlichkeiten machen die Modellkonfidenz sichtbar.
probabilities = model.predict_proba(X_test)
predicted_classes = model.predict(X_test)
confidence = np.max(probabilities, axis=1)

# Wir betrachten die drei unsichersten Testfälle.
uncertain_indices = np.argsort(confidence)[:3]
class_names = data.target_names

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Baseline-Genauigkeit: {baseline_accuracy:.3f}")
print(f"Modell-Genauigkeit: {model_accuracy:.3f}")
print("Unsicherste Vorhersagen: wahr | vorhergesagt | Konfidenz")

# Einzelne Fälle zeigen, wo menschliche Prüfung sinnvoll wäre.
for index in uncertain_indices:
    true_name = class_names[y_test[index]]
    predicted_name = class_names[predicted_classes[index]]
    value = confidence[index]
    print(f"{true_name} | {predicted_name} | {value:.3f}")



### **3.3. Keras Mini Projekt**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_15/Lecture_B/image_03_03.jpg?v=1784810361" width="250">



>* Vollständigen Modellierungsprozess mit Baselines planen
>* Daten sauber trennen und realistisch bewerten

>* Baselines setzen sinnvolle Mindestleistungen
>* Bewertung muss zur Anwendung passen

>* Overfitting vermeiden und Lernkurven prüfen
>* Vorhersagen analysieren und Modell dokumentiert speichern



In [ ]:
#@title Python-Code - Keras Mini Projekt

# Dieses Mini Projekt vergleicht ein Modell mit Baselines.
# Bewertungsdaten bleiben bis zur letzten Prüfung getrennt.
# Die Ausgabe zeigt, ob Komplexität wirklich hilft.

import sklearn
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Wir nutzen einen kleinen eingebauten Klassifikationsdatensatz.
data = load_breast_cancer()
X = data.data
y = data.target

# Eine einfache Prüfung macht die Datenannahme sichtbar.
if X.shape[0] != y.shape[0]:
    raise ValueError("Merkmale und Zielwerte passen nicht zusammen.")

# Erst trennen wir Testdaten für die faire Schlussbewertung.
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Danach trennen wir Validierungsdaten für Modellentscheidungen.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid, test_size=0.25, stratify=y_train_valid,
    random_state=42
)

# Die Baseline sagt immer die häufigste Trainingsklasse voraus.
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

# Das regulierte Modell skaliert Merkmale nur anhand der Trainingsdaten.
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=0.5, max_iter=1000, random_state=42)
)
model.fit(X_train, y_train)

# Validierungswerte helfen beim fairen Vergleich vor dem Test.
baseline_valid = balanced_accuracy_score(y_valid, baseline.predict(X_valid))
model_valid = balanced_accuracy_score(y_valid, model.predict(X_valid))

# Der Testwert wird erst nach der Modellwahl betrachtet.
test_predictions = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)
test_balanced = balanced_accuracy_score(y_test, test_predictions)

print(f"scikit-learn Version: {sklearn.__version__}")
print(f"Baseline Validierung, Balanced Accuracy: {baseline_valid:.3f}")
print(f"Modell Validierung, Balanced Accuracy: {model_valid:.3f}")
print(f"Modell Test, Accuracy: {test_accuracy:.3f}")
print(f"Modell Test, Balanced Accuracy: {test_balanced:.3f}")



# <font color="#418FDE" size="6.5" uppercase>**Dichte Keras-Modelle**</font>


In this lecture, you learned to:
- Erstellen Sequential-Modelle mit Dense-Schichten und passenden Eingabe- und Ausgabeformen. 
- Trainieren kleine Keras-Modelle mit compile, fit, Validierungsdaten und History. 
- Bewerten, regularisieren, speichern und vergleichen Keras-Modelle mit Baselines. 

In the next Module (Module 16), we will go over 'CNNs mit Keras'